# Ajuste Fino (Fine-Tuning) de Modelos Clasificadores de Texto Preentrenados

Habiendo explorado cómo construir pipelines de procesamiento de texto y clasificadores desde cero, ahora estás listo para aprovechar técnicas más avanzadas y eficientes. En lugar de entrenar un modelo desde el principio, lo cual puede ser computacionalmente intensivo y requerir enormes cantidades de datos, utilizarás un modelo preentrenado. Este enfoque utiliza un modelo que ya ha aprendido patrones de lenguaje enriquecidos a partir de conjuntos de datos masivos, dándote una ventaja poderosa a través del **aprendizaje por transferencia** (*transfer learning*).

En este laboratorio, te enfocarás en realizar el ajuste fino de **DistilBERT**, una versión más ligera y rápida del formidable modelo BERT, para clasificar títulos de recetas. Este proceso demuestra cómo adaptar un modelo de lenguaje de propósito general para una tarea especializada. También verás cómo las herramientas del ecosistema de Hugging Face simplifican muchos de los pasos manuales de preparación de datos, como la tokenización y el relleno (*padding*).

Este laboratorio te guiará a través de los siguientes pasos esenciales:

* Carga del modelo DistilBERT preentrenado junto con su tokenizador específico.
* Preparación del conjunto de datos de recetas utilizando una clase `Dataset` personalizada y un `DataCollatorWithPadding` automatizado para una creación de lotes (*batching*) eficiente.
* Implementación de dos estrategias de ajuste fino: una en la que actualizas el modelo completo y otro método más eficiente en el que solo entrenas las últimas capas.
* Comparación del rendimiento de ambos métodos para evaluar el equilibrio entre la precisión y el costo computacional.
* Prueba de tu(s) modelo(s) con títulos de recetas nuevos y desconocidos para evaluar su capacidad de generalización.

## Imports

In [ ]:
import random

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import transformers

import helper_utils

# Set random seed for reproducibility
SEED = 99
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Revisiting Recipe Dataset

Reutilizarás el conjunto de datos de recetas del laboratorio anterior. Como recordatorio, este es un subconjunto especializado del conjunto de datos masivo [Food.com Recipes and Interactions](https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions), que contiene títulos de recetas que han sido claramente clasificadas como basadas en frutas o en verduras.

### Preparación de Datos

* Carga el archivo `recipes_fruit_veg.csv` en un DataFrame de pandas.
* Crea una columna numérica `label` a partir de las categorías de texto, mapeando `'fruit'` a `0` y `'vegetable'` a `1`.
* Extrae los nombres de las recetas y las etiquetas numéricas en dos listas separadas, `texts` y `labels`.

In [ ]:
# Cargar el conjunto de datos filtrado en un DataFrame de pandas
df = pd.read_csv("recipes_fruit_veg.csv")

# Crear la columna numérica 'label': 0 para 'fruit', 1 para 'vegetable'
df['label'] = 1
df.loc[df['category'] == 'fruit', 'label'] = 0

# Extraer los nombres de las recetas y las etiquetas en listas
df_clean = df.dropna(subset=['name'])
texts = df_clean['name'].tolist()
labels = df_clean['label'].tolist()

# Verificar el tamaño del conjunto de datos y la distribución de clases
print(f"Total de muestras para clasificación: {len(texts)}")
print(f"Recetas de frutas:                     {labels.count(0)}, {round(labels.count(0)/(labels.count(0) + labels.count(1)) *100,1)} %")
print(f"Recetas de verduras:                   {labels.count(1)}, {round(labels.count(1)/(labels.count(0) + labels.count(1)) *100,1)} %")

### Previsualización de las columnas `name` y `label`

Tus datos ahora están estructurados con las columnas `name` (nombre) y `label` (etiqueta).

* Ejecuta la celda de abajo para revisar una muestra aleatoria de estos pares de entrenamiento.

In [ ]:
# Establecer el número de muestras aleatorias a mostrar.
num_samples = 10

# Mostrar una muestra de pares de nombre y etiqueta.
display(df[['name', 'label']].sample(num_samples, random_state=25).style.hide(axis="index"))

## Cargando el Transformer Preentrenado

En el laboratorio anterior, construiste cada parte de tu clasificador de texto desde cero. La diferencia fundamental en este laboratorio es que reemplazarás varios componentes que antes tenías que construir tú mismo con herramientas altamente optimizadas del ecosistema de Hugging Face.

Específicamente, utilizarás el modelo [DistilBERT](https://huggingface.co/distilbert-base-uncased). Esto implica cargar dos componentes clave que están diseñados para trabajar en conjunto:

* **El Modelo Preentrenado**: Esta es una poderosa red neuronal, DistilBERT, que ya ha aprendido a entender el lenguaje a partir de una cantidad masiva de texto. Su función es proporcionar una base sólida de comprensión del lenguaje que adaptarás para tu tarea de clasificación de recetas.
* **El Tokenizador**: Este es el puente entre tu texto bruto y el modelo. Traducirá los títulos de tus recetas al formato numérico específico con el que se entrenó el modelo. Cada modelo preentrenado tiene su propio tokenizador específico, y es crucial utilizar el que coincida con tu modelo.

* Ejecuta la celda de abajo para descargar el modelo base DistilBERT y su tokenizador desde Hugging Face.

In [ ]:
model_name="distilbert-base-uncased"
model_path="./distilbert-local-base"

# Ensure the model is downloaded
helper_utils.download_bert(model_name, model_path)

* Carga el transformer preentrenado.
    * `num_classes=2`: Conecta un nuevo cabezal de clasificación inicializado aleatoriamente con 2 etiquetas de salida, preparando el modelo para tu tarea de clasificación binaria.

**Nota**: Verás una advertencia indicando que algunos pesos fueron "recién inicializados" (*newly initialized*). Esto es normal. Confirma que has cargado con éxito la base preentrenada de DistilBERT y has añadido un nuevo cabezal de clasificación sin entrenar.

In [ ]:
bert_model, bert_tokenizer = helper_utils.load_bert(model_path, num_classes=2)

## Preparación de Datos para el Entrenamiento

Ahora que tienes tu modelo, tokenizador y listas de datos listos, el siguiente paso es estructurar esta información en los objetos que PyTorch requiere para el entrenamiento. Este proceso es más sencillo que en el laboratorio anterior porque muchos de los pasos manuales que realizaste antes, como limpiar el texto con la función `preprocess_text` y construir una clase `Vocabulary` personalizada, ya no son necesarios.

El tokenizador de Hugging Face se encarga de este trabajo por ti. Realiza la limpieza del texto, la tokenización y la conversión numérica automáticamente dentro de la clase `Dataset` personalizada que estás a punto de crear. Definirás este `Dataset` para envolver tus datos y luego usarás `DataLoaders` para crear lotes (*batches*) iterables.

### Clase de Dataset `RecipeDataset`

Comenzarás definiendo una clase `RecipeDataset`, cuyo propósito es utilizar tu tokenizador para convertir una sola muestra de texto bruto en los tensores numéricos requeridos sobre la marcha, justo cuando el modelo los necesita.

* Define el `RecipeDataset`, que servirá como contenedor para tus datos y gestionará el proceso de tokenización en tiempo real.
    * `__init__`: Inicializa el dataset almacenando tus listas de `texts`, `labels` y el `tokenizer`.
    * `__len__`: Devuelve el número total de muestras en tu dataset.
    * `__getitem__`: Este es el método central donde ocurre el procesamiento sobre la marcha. Para cada muestra de texto, la llamada al `tokenizer` realiza todos los pasos de preprocesamiento complejos que antes manejabas manualmente. Limpia el texto, lo divide en subpalabras (*sub-words*), convierte los tokens en IDs numéricos utilizando su vocabulario integrado y crea una máscara de atención (*attention mask*). El método luego combina estos tensores con la etiqueta correcta en un diccionario, listo para el modelo.

In [ ]:
class RecipeDataset(Dataset):
    """
    Dataset de PyTorch personalizado para la clasificación de texto.

    Esta clase Dataset almacena textos brutos y sus etiquetas correspondientes. Está
    diseñada para trabajar de manera eficiente con un tokenizador de Hugging Face,
    realizando la tokenización sobre la marcha para cada muestra cuando se solicita.
    """
    def __init__(self, texts, labels, tokenizer):
        """
        Inicializa el RecipeDataset.

        Args:
            texts: Una lista de cadenas de texto bruto.
            labels: Una lista de etiquetas enteras correspondientes a los textos.
            tokenizer: Una instancia de un tokenizador de Hugging Face para procesar el texto.
        """
        # Almacenar la lista de cadenas de texto bruto.
        self.texts = texts
        # Almacenar la lista de etiquetas enteras.
        self.labels = labels
        # Almacenar la instancia del tokenizador que procesará el texto.
        self.tokenizer = tokenizer

    def __len__(self):
        """Devuelve el número total de muestras en el dataset."""
        # Devolver el tamaño del dataset basado en el número de textos.
        return len(self.texts)

    def __getitem__(self, idx):
        """
        Recupera y procesa una muestra del dataset.

        Para un índice dado, este método obtiene el texto y la etiqueta correspondientes,
        tokeniza el texto y devuelve un diccionario de tensores.

        Args:
            idx: El índice de la muestra a recuperar.

        Returns:
            Un diccionario que contiene las entradas tokenizadas ('input_ids',
            'attention_mask') y las 'labels' como tensores.
        """
        # Obtener el texto bruto y la etiqueta para el índice especificado.
        text = self.texts[idx]
        label = self.labels[idx]

        # Tokenizar el texto, manejando tareas como limpieza, conversión numérica
        # y truncamiento. El relleno (padding) se maneja más tarde mediante un DataCollator.
        encoding = self.tokenizer(text, truncation=True, max_length=512)

        # Añadir la etiqueta al diccionario de codificación y convertirla en un tensor.
        encoding['labels'] = torch.tensor(label, dtype=torch.long)

        # Devolver el diccionario que contiene todos los datos procesados para la muestra.
        return encoding

* Crea una instancia de tu `RecipeDataset`.

In [ ]:
# Crear el dataset completo
full_dataset = RecipeDataset(texts, labels, bert_tokenizer)

### División de los Datos

* Divide tu `full_dataset` en un conjunto de entrenamiento del 80% y un conjunto de validación del 20%.

In [ ]:
# Dividir el conjunto de datos completo en un 80% para entrenamiento y un 20% para validación.
train_dataset, val_dataset = helper_utils.create_dataset_splits(
    full_dataset, 
    train_split_percentage=0.8
)

# Imprimir el número de muestras en cada conjunto para verificar la división.
print(f"Muestras de entrenamiento: {len(train_dataset)}")
print(f"Muestras de validación:    {len(val_dataset)}")

### Crear DataLoaders

En el laboratorio anterior, abordaste el desafío de agrupar textos de longitud variable escribiendo funciones `collate_fn` personalizadas para rellenar (*pad*) secuencias manualmente o crear desplazamientos (*offsets*). La función `DataCollatorWithPadding` de Hugging Face automatiza este complejo paso por ti.

* Utiliza [DataCollatorWithPadding](https://huggingface.co/docs/transformers/en/main_classes/data_collator#transformers.DataCollatorWithPadding) y pásale tu `bert_tokenizer`. Este se encargará automáticamente del relleno dinámico de cada lote.

In [ ]:
# El colector de datos (data collator) maneja el relleno dinámico para cada lote.
data_collator = transformers.DataCollatorWithPadding(tokenizer=bert_tokenizer)

* Crea dos instancias de `DataLoader`: `train_loader` y `val_loader`.
    * `collate_fn=data_collator`: Pasa tu `data_collator` para crear lotes con relleno dinámico en lugar de utilizar el comportamiento predeterminado de PyTorch.

In [ ]:
# Establecer el número de muestras a procesar en cada lote.
batch_size = 32

# Crear el DataLoader para el conjunto de entrenamiento con el `data_collator`
train_loader = DataLoader(train_dataset, 
                          batch_size=batch_size, 
                          shuffle=True, 
                          collate_fn=data_collator
                         )

# Crear el DataLoader para el conjunto de validación con el `data_collator`
val_loader = DataLoader(val_dataset, 
                        batch_size=batch_size, 
                        shuffle=False, 
                        collate_fn=data_collator
                       )

## Entrenamiento del Modelo

Con el modelo DistilBERT preentrenado cargado y los `DataLoaders` totalmente configurados, el trabajo de base está terminado. Ahora estás listo para comenzar el proceso de ajuste fino (*fine-tuning*).

### Abordando el Desequilibrio de Clases

* Calcula los pesos de las clases para solucionar el desequilibrio de datos en tu conjunto de entrenamiento.

In [ ]:
# Extraer todas las etiquetas del conjunto de entrenamiento para calcular los pesos de clase.
train_labels_list = [train_dataset.dataset.labels[i] for i in train_dataset.indices]

# Usar la utilidad de scikit-learn para calcular automáticamente los pesos de clase.
class_weights = compute_class_weight(
    # La estrategia para calcular los pesos. 'balanced' es automática.
    class_weight='balanced',
    # El array de etiquetas de clase únicas (por ejemplo, [0, 1]).
    classes=np.unique(train_labels_list),
    # La lista de todas las etiquetas de entrenamiento, usada para contar frecuencias.
    y=train_labels_list
)

# Convertir el array de NumPy de los pesos en un tensor de PyTorch de tipo float.
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Imprimir los pesos finales para verificar el cálculo.
print("Pesos de Clase Calculados:")
print(f"  - Fruta (Clase 0):     {class_weights[0]:.2f}")
print(f"  - Verdura (Clase 1):   {class_weights[1]:.2f}")

### Configuración de la Función de Pérdida

* Define `nn.CrossEntropyLoss` como tu función de pérdida y pasa tu tensor `class_weights` calculado previamente al parámetro `weight`.

In [ ]:
# Inicializar la función CrossEntropyLoss con los `class_weights` calculados.
loss_function = nn.CrossEntropyLoss(weight=class_weights)

### Baseline Approach: Fine-Tuning de todo el modelo

Primero, tomarás el enfoque estándar: realizar el **fine-tuning** de *todo* el modelo DistilBERT. Esto significa que cada parámetro, desde las capas de **embedding** iniciales hasta la capa de clasificación final, verá sus pesos actualizados durante el entrenamiento. Ten en cuenta que comenzaremos el entrenamiento utilizando los pesos pre-trained y continuaremos entrenando el modelo a partir de ahí.

Este método adapta el modelo completo a la tarea de clasificación de recetas y servirá como tu **performance baseline**. Utilizarás la función `training_loop` para ejecutar el proceso de entrenamiento y ver qué tan bien funciona este enfoque.

* Para cada **batch**, la función descompone explícitamente los `input_ids`, `attention_mask` y `labels` requeridos por el modelo.
* Luego, realiza el **fine-tuning** de todas las capas del modelo DistilBERT con tu dataset.

In [ ]:
## Descomentar si quieres ver la función del bucle de entrenamiento

# helper_utils.display_function(helper_utils.training_loop)

In [ ]:
# Establecer el número total de epochs.
num_epochs = 3

# Llamar al training loop para iniciar el proceso de full fine-tuning.
full_finetuned_bert, full_results = helper_utils.training_loop(
    bert_model, 
    train_loader, 
    val_loader, 
    loss_function, 
    num_epochs, 
    device
)

* Imprime las métricas de validación del diccionario `results_bert` para revisar el rendimiento de tu modelo con **fine-tuning** en el conjunto de validación.

In [ ]:
# Display the results 
helper_utils.print_final_results(full_results)

### Una alternativa eficiente: Fine-Tuning parcial

Aunque realizar el **fine-tuning** de todo el modelo es efectivo, puede ser computacionalmente costoso. Ahora, explorarás una estrategia más eficiente conocida como **fine-tuning parcial**. En lugar de entrenar el modelo completo, congelarás (*freeze*) estratégicamente la mayoría de las capas del modelo y entrenarás solo aquellas que son más efectivas para adaptarse a la nueva tarea.

Para asegurar una comparación justa entre los dos enfoques, primero debes volver a cargar el modelo DistilBERT original **pre-trained**, ya que el bucle de entrenamiento anterior actualizó los pesos del modelo *in-place*.

**Nota**: Verás una advertencia indicando que algunos pesos fueron "recién inicializados" (*newly initialized*). Esto es normal. Confirma que has cargado con éxito la base **pre-trained** de DistilBERT y has añadido un nuevo cabezal de clasificación sin entrenar.


In [ ]:
# RECARGAR el modelo base para asegurar una comparación justa
bert_model, bert_tokenizer = helper_utils.load_bert(model_path, num_classes=2)

<br>

Para llevar a cabo el **fine-tuning** parcial, necesitas identificar las capas específicas que vas a congelar y las que vas a entrenar. Primero, comienza inspeccionando la arquitectura del modelo DistilBERT.

In [ ]:
print(bert_model)

<br>

La decisión de qué capas congelar se basa en cómo los **transformers** aprenden de forma jerárquica:

* **Earlier Layers**: 
Las capas más cercanas a la entrada aprenden características generales del lenguaje, como la gramática y las relaciones básicas entre palabras. 
Dado que estas características son útiles para casi cualquier tarea, a menudo se mantienen congeladas (**frozen**). 
En tu modelo DistilBERT, estas son los `embeddings` y las **primeras cuatro** capas de `TransformerBlock`:

In [ ]:
# embeddings
print("\nEmbeddings: \n")
print(bert_model.distilbert.embeddings)

# primeras cuatro capas TransformerBlock
print("\nFirst four TransformerBlock layers: \n")
print(bert_model.distilbert.transformer.layer[:4])

<br>

* **Later Layers**: Las capas más cercanas a la salida aprenden características más complejas y abstractas que se vuelven más especializadas en los datos con los que se entrenan. Estas son las capas que normalmente querrás descongelar (**unfreeze**) para adaptar el modelo a los matices de tu nueva tarea. En tu modelo, estas son las **últimas dos** capas de `TransformerBlock` y las capas finales de clasificación:

In [ ]:
# últimas dos capas TransformerBlock
print("\nLast two TransformerBlock layers: \n")
print(bert_model.distilbert.transformer.layer[4:6])

# capas de clasificación finales
print("\nFinal Classifier Layer: \n")
print(bert_model.pre_classifier)
print(bert_model.classifier)

<br>

Para la tarea en cuestión, vas a descongelar (**unfreeze**) y entrenar el **cabezal de clasificación final** y las **últimas dos capas del transformer** (**later layers**). Esto permite que el modelo ajuste su extracción de características de alto nivel a los matices de la clasificación de recetas, mientras sigue aprovechando la robusta comprensión general del lenguaje de sus capas congeladas.

Este enfoque pone a prueba una hipótesis clave: ¿puedes lograr un rendimiento comparable al **baseline** ahorrando recursos computacionales significativos?

* Tu primer paso es congelar todos los parámetros del modelo estableciendo su atributo `requires_grad` en `False`. Esto evita que sus pesos se actualicen durante el proceso de entrenamiento.

In [ ]:
# Freeze ALL model parameters first
for param in bert_model.parameters():
    param.requires_grad = False

* A continuación, descongelarás las **últimas dos capas del transformer** para que sean entrenables, volviendo a establecer su atributo `requires_grad` en `True`.

In [ ]:
# Descongelar las últimas 2 capas del transformer
# Establecer el número de capas finales del transformer a descongelar y entrenar.
layers_to_train = 2 

# Acceder a la lista de todas las capas del transformer en el modelo DistilBERT.
transformer_layers = bert_model.distilbert.transformer.layer

# Recorrer hacia atrás desde el final de la lista de capas para el número de capas que deseas entrenar.
for i in range(layers_to_train):
    # Seleccionar una capa usando indexación negativa (ej., -1 para la última, -2 para la penúltima).
    layer_to_unfreeze = transformer_layers[-(i+1)]
    
    # Iterar a través de todos los parámetros de la capa seleccionada.
    for param in layer_to_unfreeze.parameters():
        # Establecer requires_grad en True para hacer que el parámetro sea entrenable.
        param.requires_grad = True

* El paso final es descongelar el cabezal de clasificación del modelo, que consiste en las capas `pre_classifier` y `classifier`, para asegurar que pueda ser entrenado en tu nueva tarea.

In [ ]:
# Descongelar el cabezal del clasificador
# Las capas finales del modelo deben hacerse entrenables para adaptarse a la nueva tarea.

# Para DistilBERT, este cabezal consiste en dos capas lineales.
# Descongelar la capa pre_classifier.
for param in bert_model.pre_classifier.parameters():
    param.requires_grad = True

# Descongelar la capa final classifier.
for param in bert_model.classifier.parameters():
    param.requires_grad = True

Con tu estrategia de **partial fine-tuning** configurada, ahora ejecutarás la función `training_loop` para iniciar el proceso de entrenamiento. Esta se encargará de todo el proceso y devolverá el modelo entrenado junto con un diccionario de las métricas de validación finales.

* Para cada **batch**, descompone explícitamente los `input_ids`, `attention_mask` y `labels` requeridos por el modelo.

In [ ]:
## Descomentar si quieres ver la función del bucle de entrenamiento

# helper_utils.display_function(helper_utils.training_loop)

In [ ]:
# Establecer el número total de epochs.
num_epochs = 3

# Llamar al training loop para iniciar el proceso de partial fine-tuning.
partial_finetuned_bert, partial_results = helper_utils.training_loop(
    bert_model, 
    train_loader, 
    val_loader, 
    loss_function, 
    num_epochs, 
    device
)

* Imprime las métricas de validación del diccionario `results_bert` para revisar el rendimiento de tu modelo con **fine-tuning** en el conjunto de validación.

In [ ]:
# Display the results 
helper_utils.print_final_results(partial_results)

### Comparación de los Enfoques de Fine-Tuning

* Compara directamente el rendimiento de los dos enfoques: el **baseline** de **full fine-tuning** y el método eficiente de **partial fine-tuning**.
    * `full_results`: Contiene las métricas obtenidas tras realizar el **full fine-tuning** de todo el modelo.
    * `partial_results`: Contiene las métricas del **partial fine-tuning**, donde solo se entrenaron las últimas dos capas del transformer y el clasificador.

In [ ]:
# Compare your results
helper_utils.display_results(full_results, partial_results)

<br>

Basándote en estos resultados, el modelo con **partial fine-tuning** rinde casi tan bien como el **baseline** de **full fine-tuning** después de solo `3` **epochs**. Aunque el enfoque de **full fine-tuning** resultó en métricas ligeramente superiores, la estrategia de **partial fine-tuning** demostró ser altamente efectiva. Como habrás notado, este enfoque más eficiente actualizó muchos menos parámetros, ahorrando tiempo de entrenamiento. Esto ilustra el beneficio principal del **partial fine-tuning**: lograr un rendimiento competitivo reduciendo significativamente los costos computacionales.

## Probando el modelo BERT con Fine-tuning en nuevos ejemplos

Ahora la prueba final. Es momento de ver cómo se comporta tu modelo con **fine-tuning** ante datos completamente nuevos que nunca ha visto. Esta es la mejor forma de obtener una sensación cualitativa de qué tan bien ha aprendido el modelo a generalizar.

* Define una lista `test_products` que contenga una mezcla de nuevos títulos de recetas. Esta lista incluye ejemplos directos, así como otros más desafiantes, para ver dónde sobresale el modelo y dónde podría tener dificultades.
    * ¡Siéntete libre de añadir tus propios títulos de recetas a esta lista para poner a prueba el modelo aún más!

**Nota**: Recuerda que las predicciones del modelo se basan *únicamente* en las palabras del `name` de la receta. Nunca se le mostró la lista de ingredientes, por lo que no tiene conocimiento de si las frutas o las verduras son el ingrediente dominante. El nombre de una receta a veces puede ser engañoso, y la clasificación del modelo reflejará solo lo que ha aprendido del texto del título.

In [ ]:
test_products = [
    "Blueberry Muffins",                  # Expected: Fruit
    "Spinach and Feta Stuffed Chicken",   # Expected: Vegetable
    "Classic Carrot Cake with Frosting",  # Expected: Vegetable
    "Tomato and Basil Bruschetta",        # Expected: Vegetable
    "Avocado Toast",                      # Expected: Fruit
    "Zucchini Bread with Walnuts",        # Expected: Vegetable
    "Lemon and Herb Roasted Chicken",     # Expected: Fruit
    "Strawberry Rhubarb Pie",             # Expected: Fruit
]


* Finalmente, recorre la lista `test_products` con un bucle para ejecutar la predicción de cada receta y observar el **output** final del modelo.

In [ ]:
## Descomentar si quieres ver la función de predicción de categoría

# helper_utils.display_function(helper_utils.predict_category)

In [ ]:
# Recorrer cada producto de prueba
for product in test_products:
    # Llamar a la función de predicción con los argumentos requeridos
    category = helper_utils.predict_category(
        partial_finetuned_bert, # Pruébalo también con `full_finetuned_bert`.
        bert_tokenizer,
        product,
        device
    )
    # Imprimir los resultados
    print(f"Producto: '{product}'\nPredicción: {category}.\n")

## Conclusion

¡Felicitaciones por completar este laboratorio! Has pasado con éxito de construir modelos desde cero a realizar el **fine-tuning** de un modelo **transformer** de vanguardia para una tarea personalizada de clasificación de texto.

Comenzaste cargando un modelo DistilBERT **pre-trained** y viste de primera mano cómo simplifica todo el **pipeline** de texto a tensor. La principal lección de tus experimentos es la efectividad del **partial fine-tuning**. Demostraste que, al congelar estratégicamente la mayoría de las capas del modelo y actualizar solo las finales (específicas para la tarea), puedes lograr un rendimiento comparable al **full fine-tuning** de todo el modelo. Este conocimiento es inmensamente valioso para aplicaciones prácticas, ya que permite ahorros significativos en tiempo de entrenamiento y recursos computacionales sin sacrificar la calidad.

Las habilidades que has desarrollado aquí —cargar y adaptar modelos **pre-trained**, gestionar datos con herramientas modernas y elegir estratégicamente qué partes de un modelo entrenar— son los pilares para abordar una amplia gama de desafíos complejos de **NLP**, desde el análisis de sentimiento hasta la traducción automática y más allá.